---
## Save the center of mass (CoM) of the grinder from the FRCNN tracks
Saves the grinder CoM for all frames of all videos. **Make sure that you have run `EZ_FRCNN.ipynb` to track the grinder first!**

In [ ]:
### Import libraries + define paths
from library import *
from skimage import color
from skimage.util import img_as_ubyte
from skimage.exposure import equalize_hist
from scipy.ndimage import center_of_mass

VID_DIR = './videos/'  # Directory containing videos to be analyzed
OUT_DIR = './outputs/' # Directory where all code outputs (FRCNN tracks, plots) are stored

### Get names of all videos within the folder
video_type  = '.wmv' # CHANGE FOR YOUR DATA
video_paths = glob.glob(VID_DIR + '*' + video_type)  
videos      = [os.path.splitext(os.path.basename(v))[0] for v in video_paths]

In [ ]:
### Calculate COMs for all videos in recording
FILTER = True
for video in videos:
    print('---')
    print('Processing ' + video + + video_type + '...')
    ### Load data
    fullName    = video + '_cropped' + video_type
    vidPath     = VID_DIR + '/cropped/' + fullName

    grinderName = VID_DIR + '/outputs/frcnn_grinder/' + video + '_cropped_boxes.csv'
    grinders    = pd.read_csv(grinderName)
    grinders    = grinders.to_numpy()

    ### Load video
    vid        = cv2.VideoCapture(vidPath)
    tot_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))
    
    ### Main loop for calculating and saving grinder CoM
    coms = np.zeros((tot_frames,2))
    success = True
    i = 0
    while success and i < tot_frames-1:
        success, frame = vid.read()

        if success and np.sum(grinders[i]) > 0:
            # Calculate centroid of grinder in bounding box
            cropped = frame[int(grinders[i][1]): int(grinders[i][3]),int(grinders[i][0]): int(grinders[i][2])]
            box = color.rgb2gray(cropped) # grayscale
            box = img_as_ubyte(box)
            box = equalize_hist(box)  # equalize (increase grinder contrast)
            invert_box = 1 - box      # invert
            coms[i] = center_of_mass(invert_box)

            # Convert coordinates back to 250x250
            coms[i][0] += grinders[i][0]
            coms[i][1] += grinders[i][1]

        # Update counter
        i += 1
        
    # (OPTIONAL) Filter CoMs to fill gaps from dropped frames
    if FILTER:
        filtered_coms, _ = filter_ROIs(coms)
    
        # Save CoMs
        save_path = VID_DIR + '/outputs/coms/' + video + '_coms.csv'
        np.savetxt(save_path,filtered_coms)
        print('Done. Saved to ' + save_path)
    
    else:
        # Save CoMs
        save_path = VID_DIR + '/outputs/coms/' + video + '_coms.csv'
        np.savetxt(save_path,coms)
        print('Done. Saved to ' + save_path)

print('---')